# v0.25.0 — Embedded engine and test fixtures

## 1. Intro

The official surrealdb SDK can run SurrealDB **inside your Python process**: no server, no Docker.
Since v0.25.0 the ORM runs on that in-process engine. Point `set_connection()` at `mem://` (or at an
on-disk `surrealkv://<path>`) and the same models, queries, relations, transactions and change feeds
work unchanged. `surreal_orm_lite.testing` adds the test helpers: `embedded_database()`, an isolated
database per block, and the full SurrealDB-ORM's `SurrealFixture` / `@fixture` for declarative test
data.

What the engine does not do, each refused up front with a clear ORM error:

* **live queries**: it has no notification channel. Read the table's change feed instead;
* **model auth on a session of its own** (`bind=False`) and `ephemeral_client()`: it has one session
  per database;
* **`changes(since=<datetime>)`**: pass a cursor;
* **`rocksdb://`**: the official SDK does not accept it yet
  ([#233](https://github.com/EulogySnowfall/SurrealDB-ORM-lite/issues/233)).

The engine bundled with SDK 2.0.0 is SurrealDB core **2.3.10**: its SurrealQL is the 2.x line,
whichever server line you deploy. Every section but the last runs without a server; section 12 runs
the same code on a server when one is reachable.

## 2. Connect

An embedded URL needs only a namespace and a database. `user` and `password` are optional there and
never sent: the anonymous session of the engine is the in-process owner. Nothing opens yet: the first
ORM call starts the engine.

In [1]:
import asyncio
import contextlib
import os

from surreal_orm_lite import SurrealDBConnectionManager

SurrealDBConnectionManager.set_connection(url="mem://", namespace="examples", database="examples")
print("Connection configured:", SurrealDBConnectionManager.is_connection_set())
print("Embedded engine:      ", SurrealDBConnectionManager.is_embedded())

Connection configured: True
Embedded engine:       True


The ORM keeps no list of embedded schemes: the SDK decides. A URL the installed SDK refuses, such as
`rocksdb://` with SDK 2.0.0, is accepted by `set_connection()` and reported by the first ORM call,
before anything is opened, with a `SurrealDbConnectionError` that quotes the SDK.

In [2]:
from surreal_orm_lite import SurrealDbConnectionError

SurrealDBConnectionManager.set_connection(url="rocksdb://example.db", namespace="examples", database="examples")
try:
    await SurrealDBConnectionManager.get_client()
except SurrealDbConnectionError as error:
    print(error)

# Back to the in-memory engine.
SurrealDBConnectionManager.set_connection(url="mem://", namespace="examples", database="examples")
print("\nEmbedded engine:", SurrealDBConnectionManager.is_embedded())

The installed surrealdb SDK cannot open 'rocksdb://example.db': 'rocksdb' is not a valid UrlScheme. The URL schemes it accepts are: file, http, https, mem, memory, surrealkv, surrealkv+versioned, ws, wss. Embedded RocksDB (rocksdb://) is not supported by the official SDK yet: see https://github.com/EulogySnowfall/SurrealDB-ORM-lite/issues/233.

Embedded engine: True


For raw SurrealQL, a small helper asks the manager for the client on every call instead of keeping
it in a variable. Section 10 explains why that matters on the embedded engine.

In [3]:
async def query(sql: str, variables: dict | None = None):
    """Run raw SurrealQL on the configured connection, without keeping its client."""
    client = await SurrealDBConnectionManager.get_client()
    return await client.query(sql, variables or {})


print("2.x SurrealQL works (type::is::string):", await query("RETURN type::is::string('ok');"))

2.x SurrealQL works (type::is::string): True


## 3. Models

Plain ORM models, nothing specific to the engine: an author with a birth date (a rich field type,
v0.23.0), a book with the point where its first edition was published (a geometry field, v0.24.0),
and a typed `wrote` edge between them.

In [4]:
from datetime import date

from surreal_orm_lite import BaseSurrealEdge, BaseSurrealModel, PointField, SurrealConfigDict


class Author(BaseSurrealModel):
    model_config = SurrealConfigDict(table_name="author")

    id: str | None = None
    name: str
    born: date


class Book(BaseSurrealModel):
    model_config = SurrealConfigDict(table_name="book")

    id: str | None = None
    title: str
    year: int
    pages: int
    genre: str = ""
    location: PointField = None  # where the first edition was published


class Wrote(BaseSurrealEdge):
    model_config = SurrealConfigDict(table_name="wrote")


print("tables:", Author.get_table_name(), Book.get_table_name(), Wrote.get_table_name())

tables: author book wrote


## 4. CRUD, QuerySet and aggregation

The scenario is a function, so that section 12 can run the very same code on a server and compare
the results. It saves three authors and four books, creates, updates and deletes a draft whose id
SurrealDB picks, then filters, orders and aggregates. The writes are sequential: section 11 says
what to do before concurrent writes on a fresh engine.

In [5]:
from surreal_orm_lite import Avg, Count, GeometryPoint


async def catalogue() -> dict[str, object]:
    """Save the authors and books, then query them: the same code on any connection."""
    for key, name, born in [
        ("tolkien", "J. R. R. Tolkien", date(1892, 1, 3)),
        ("le_guin", "Ursula K. Le Guin", date(1929, 10, 21)),
        ("herbert", "Frank Herbert", date(1920, 10, 8)),
    ]:
        await Author(id=key, name=name, born=born).save()
    for key, title, year, pages, genre, location in [
        ("hobbit", "The Hobbit", 1937, 310, "fantasy", (-0.1276, 51.5072)),  # London
        ("earthsea", "A Wizard of Earthsea", 1968, 205, "fantasy", (-122.2727, 37.8716)),  # Berkeley
        ("dune", "Dune", 1965, 412, "science fiction", (-75.1652, 39.9526)),  # Philadelphia
        ("dispossessed", "The Dispossessed", 1974, 341, "science fiction", (-74.006, 40.7128)),  # New York
    ]:
        await Book(id=key, title=title, year=year, pages=pages, genre=genre, location=location).save()

    draft = await Book(title="Draft", year=2026, pages=10).save()  # no id: SurrealDB picks one
    await draft.merge(pages=12)
    reread = await Book.objects().get(draft.get_id())
    await reread.delete()

    born_before_1925 = Author.objects().filter(born__lt=date(1925, 1, 1)).order_by("born")
    per_genre = Book.objects().values("genre").annotate(books=Count(), pages=Avg("pages"))
    return {
        "draft pages after merge()": reread.pages,
        "books after the draft's delete()": await Book.objects().count(),
        "authors born before 1925": [a.name for a in await born_before_1925.exec()],
        "books, newest first": [b.title for b in await Book.objects().order_by("-year").exec()],
        "fantasy after 1950": [b.title for b in await Book.objects().filter(genre="fantasy", year__gt=1950).exec()],
        "first by title": (await Book.objects().order_by("title").first()).title,
        "average pages": await Book.objects().avg("pages"),
        "per genre (books, mean pages)": {row["genre"]: (row["books"], row["pages"]) for row in await per_genre.exec()},
    }


embedded_results = await catalogue()
width = max(map(len, embedded_results)) + 1
for label, value in embedded_results.items():
    print(f"{label + ':':<{width}} {value}")

draft pages after merge():        12
books after the draft's delete(): 4
authors born before 1925:         ['J. R. R. Tolkien', 'Frank Herbert']
books, newest first:              ['The Dispossessed', 'A Wizard of Earthsea', 'Dune', 'The Hobbit']
fantasy after 1950:               ['A Wizard of Earthsea']
first by title:                   A Wizard of Earthsea
average pages:                    317.0
per genre (books, mean pages):    {'fantasy': (2, 257.5), 'science fiction': (2, 376.5)}


## 5. Relations and geometry

`relate()` creates an edge from a node; an edge saved on its own goes through the SDK's
`insert_relation()`. Traversals and `get_related()` read them back. `nearby()` and `GeoDistance` work
on the engine as on a server.

In [6]:
from surrealdb import RecordID

tolkien = await Author.objects().get("tolkien")
hobbit = await Book.objects().get("hobbit")
edge = await tolkien.relate(Wrote, hobbit)
print("relate():          ", edge.in_, "->", edge.out)

for author_id, book_id in [("le_guin", "earthsea"), ("le_guin", "dispossessed"), ("herbert", "dune")]:
    await Wrote(in_=RecordID("author", author_id), out=RecordID("book", book_id)).save()
print("wrote edges:       ", await Wrote.objects().count())

le_guin = await Author.objects().get("le_guin")
print("->wrote->book:     ", sorted(str(record_id) for record_id in await le_guin.traverse("->wrote->book")))
writers = await hobbit.get_related(Wrote, direction="in", model_class=Author)
print("The Hobbit, by:    ", [author.name for author in writers])

relate():           author:tolkien -> book:hobbit
wrote edges:        4
->wrote->book:      ['book:dispossessed', 'book:earthsea']
The Hobbit, by:     ['J. R. R. Tolkien']


In [7]:
from surreal_orm_lite import GeoDistance

new_york = GeometryPoint(-74.006, 40.7128)
near = await (
    Book.objects()
    .nearby("location", new_york, max_distance=200_000)
    .annotate(dist=GeoDistance("location", new_york))
    .order_by("dist")
    .exec()
)
print("first published within 200 km of New York:", [(b.title, f"{b.dist / 1000:.0f} km") for b in near])

first published within 200 km of New York: [('The Dispossessed', '0 km'), ('Dune', '130 km')]


## 6. Transactions are buffered

The SDK refuses interactive transactions on the engine, so `transaction()` picks the **buffered**
strategy, as over HTTP: the writes are queued and sent as one atomic batch at the end of the block,
or dropped when the block raises. Inside the block, `save(tx=)` needs an explicit id and reads raise.

In [8]:
async with SurrealDBConnectionManager.transaction() as tx:
    print("strategy:", type(tx).__name__, "| interactive:", tx.is_interactive)
    await Book(id="left_hand", title="The Left Hand of Darkness", year=1969, pages=286, genre="science fiction").save(tx=tx)
    await Book(id="lathe", title="The Lathe of Heaven", year=1971, pages=184, genre="science fiction").save(tx=tx)
print("after the commit:  ", await Book.objects().count(), "books")


class Abort(Exception):
    """Raised on purpose to roll the transaction back."""


try:
    async with SurrealDBConnectionManager.transaction() as tx:
        await Book(id="messiah", title="Dune Messiah", year=1969, pages=256, genre="science fiction").save(tx=tx)
        raise Abort
except Abort:
    pass
print("after the rollback:", await Book.objects().count(), "books")

strategy: BufferedTransaction | interactive: False
after the commit:   6 books
after the rollback: 6 books


## 7. Change feeds instead of live queries

The engine has no channel to push live notifications, so `live()`, `watch()` and `subscribe_live()`
are refused. A table's **change feed** works: define the table with `CHANGEFEED` before its first
write, then read it with `changes()`, from now or from a saved cursor. A change feed records no
creation: the first save of a record comes out as an `UPDATE`, as on a server. `asyncio.timeout`
keeps a quiet stream from hanging the notebook.

In [9]:
class Loan(BaseSurrealModel):
    model_config = SurrealConfigDict(table_name="book_event")

    id: str | None = None
    book: str
    status: str


await query("DEFINE TABLE book_event CHANGEFEED 1h;")  # before any write to the table

stream = Loan.objects().changes(poll_interval=0.05)
await stream.start()
loan = await Loan(id="loan1", book="dune", status="borrowed").save()
await loan.merge(status="returned")

events = []
async with asyncio.timeout(5):
    while len(events) < 2:
        events.append(await anext(stream))
stream.stop()
for event in events:
    print(event.action, event.record_id, event.instance.status)

UPDATE book_event:loan1 borrowed
UPDATE book_event:loan1 returned


In [10]:
from surreal_orm_lite import SurrealDbError

try:
    async with Book.objects().live() as live_stream:
        pass
except SurrealDbError as error:
    print(error)

Live queries need a WebSocket server connection (ws:// or wss://): the embedded engine ('mem://') has no notification channel to deliver them. Read the table's change feed with Model.objects().changes() instead, or connect to a server.


## 8. One engine per process

A server client is bound to the event loop that opened it, so the manager keeps one per loop. The
engine is not: the manager keeps **one engine for the whole process**, shared by every event loop
and thread, so they all see the same database. Here a thread runs its own event loop and counts the
same books.

In [11]:
async def count_books() -> int:
    return await Book.objects().count()


here = await count_books()
elsewhere = await asyncio.to_thread(lambda: asyncio.run(count_books()))
print("this event loop:", here, "| another loop, on another thread:", elsewhere)

this event loop: 6 | another loop, on another thread: 6


## 9. Authentication on the engine

Record access methods, `DEFINE USER` and table permissions work on the engine. It has a **single
session**, shared by the whole process, so model auth signs that session in: pass `bind=True`.
`info()` then returns the signed-in record, and `invalidate()` goes back to the in-process owner,
for whom `info()` is `None`.

In [12]:
from surreal_orm_lite import AuthenticatedUserMixin


class User(AuthenticatedUserMixin, BaseSurrealModel):
    model_config = SurrealConfigDict(table_name="user", access_name="account")

    id: str | None = None
    email: str
    password: str


applied = await User.define_access()
print("access defined:", len(applied), "statements")

result = await User.signup(email="ada@example.test", password="s3cret-passphrase", bind=True)
print("signed up:", type(result.user).__name__, result.user.email)
print("the shared session is the user's:", SurrealDBConnectionManager.get_session_token() == result.tokens.access)
print("info():", (await SurrealDBConnectionManager.info())["email"])

await SurrealDBConnectionManager.invalidate()
print("after invalidate(), info():", await SurrealDBConnectionManager.info())

access defined: 2 statements


signed up: User ada@example.test
the shared session is the user's: True
info(): ada@example.test
after invalidate(), info(): None


Without `bind=True`, a model-auth call would need a session of its own, which the engine cannot
open: the ORM says so before sending anything.

In [13]:
try:
    await User.signin(email="ada@example.test", password="s3cret-passphrase")
except SurrealDbError as error:
    print(error)

User.signin() runs on a session of its own so that the shared connection keeps its identity, and the embedded engine ('mem://') has a single session per database. Pass bind=True to sign that session in (every model then runs as this user, in every event loop of the process), or connect to a server.


## 10. An on-disk database

`surrealkv://<path>` stores the database in a directory: closing the connection closes the engine,
and the next ORM call reopens the same path with its data. Switching to another URL also closes the
in-memory engine of the previous sections, and its data with it.

**Do not keep a client across a close.** With the surrealdb SDK 2.x, closing does not stop an
in-process engine: a client obtained before `close_connection()` (or inside an
`embedded_database()` block) is still a running database. Do not use it after the close, and drop it
(`del client`) before reopening the same `surrealkv://` path: two engines on one path miss each
other's writes. That is why this notebook never keeps a client in a variable (the `query()` helper
asks for it on each call). Never open one path from two processes.

In [14]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as directory:
    SurrealDBConnectionManager.set_connection(
        url=f"surrealkv://{Path(directory) / 'library'}", namespace="examples", database="examples"
    )
    await Author(id="butler", name="Octavia E. Butler", born=date(1947, 6, 22)).save()
    await SurrealDBConnectionManager.close_connection()  # the engine closes; its files stay
    reopened = await Author.objects().get("butler")  # the next call reopens the same path
    print("read back after the close:", reopened.name)
    await SurrealDBConnectionManager.close_connection()  # before the directory is removed
print("temporary directory removed")

SurrealDBConnectionManager.set_connection(url="mem://", namespace="examples", database="examples")
print("books in a new in-memory database:", await Book.objects().count())

read back after the close: Octavia E. Butler


temporary directory removed


books in a new in-memory database: 0


## 11. Test fixtures

`embedded_database()` runs a block against a fresh, isolated in-memory database (`mem://`,
namespace and database `test` by default), then puts the previous connection back: its settings,
stored session and engine. It yields the block's SDK client for raw SurrealQL and DDL, valid
**inside the block only**.

In [15]:
from surreal_orm_lite.testing import SurrealFixture, embedded_database, fixture

await Author(id="shelley", name="Mary Shelley", born=date(1797, 8, 30)).save()

async with embedded_database() as client:
    print("inside the block:", await Author.objects().count(), "authors")
    await client.query("DEFINE TABLE author SCHEMALESS;")
    await Author(id="wells", name="H. G. Wells", born=date(1866, 9, 21)).save()
    print("inside the block, after a save:", [a.name for a in await Author.objects().exec()])
del client  # valid inside the block only

print("after the block:", [a.name for a in await Author.objects().exec()])

inside the block: 0 authors
inside the block, after a save: ['H. G. Wells']
after the block: ['Mary Shelley']


`SurrealFixture` and `@fixture` (the full SurrealDB-ORM's names) declare model instances as test
data. `load()` saves a fresh copy of each, in declaration order, yields them and deletes them on
exit; the templates are never modified. Unlike the full ORM, an explicit `id` is kept, so an edge can
name its endpoints, and a load that fails half-way deletes what it saved. With pytest, wrap
`embedded_database()` in a fixture that yields nothing (see the README) for a fresh database per
test. Before concurrent writes (`asyncio.gather` of saves) on a fresh engine, define the table first
(`DEFINE TABLE`), as above: concurrent first writes to a table that does not exist yet raise a
retryable write conflict.

In [16]:
@fixture
class Library(SurrealFixture):
    herbert = Author(id="herbert", name="Frank Herbert", born=date(1920, 10, 8))
    dune = Book(id="dune", title="Dune", year=1965, pages=412, genre="science fiction")
    wrote = Wrote(in_=RecordID("author", "herbert"), out=RecordID("book", "dune"))
    verne = Author(name="Jules Verne", born=date(1828, 2, 8))  # no id: SurrealDB picks one


async with embedded_database():
    async with Library.load() as library:
        print("loaded:", library.herbert.name, "|", library.dune.title, "|", library.verne.name)
        print("an id was assigned:", library.verne.get_id() is not None, "| template untouched:", Library.verne.id is None)
        writers = await library.dune.get_related(Wrote, direction="in", model_class=Author)
        print("Dune, by:", [author.name for author in writers])
    print("after load():", await Author.objects().count(), "authors,", await Wrote.objects().count(), "edges")

loaded: Frank Herbert | Dune | Jules Verne
an id was assigned: True | template untouched: True
Dune, by: ['Frank Herbert']
after load(): 0 authors, 0 edges


## 12. The same code on a server (optional)

Section 4's scenario, unchanged, on a SurrealDB server over WebSocket (`SURREALDB_HOST` and
`SURREALDB_PORT`, default `localhost:8000`). Only the URL and the credentials differ. Without a
server the cell says so and the notebook goes on.

In [17]:
import socket

HOST = os.environ.get("SURREALDB_HOST", "localhost")
PORT = os.environ.get("SURREALDB_PORT", "8000")


async def drop_tables() -> None:
    """Drop the scenario's tables, whether or not they exist."""
    for table in ("author", "book"):
        with contextlib.suppress(Exception):
            await query(f"REMOVE TABLE {table};")


try:
    with socket.create_connection((HOST, int(PORT)), timeout=2):
        pass
    SurrealDBConnectionManager.set_connection(
        url=f"ws://{HOST}:{PORT}/rpc", user="root", password="root", namespace="examples", database="examples"
    )
    await SurrealDBConnectionManager.get_client()
except Exception:  # nothing listening there, or the server refused the connection
    print("no server reachable")
else:
    await drop_tables()
    server_results = await catalogue()
    await drop_tables()
    print("embedded engine:", SurrealDBConnectionManager.is_embedded())
    print("same results as on the embedded engine:", server_results == embedded_results)

embedded engine: False
same results as on the embedded engine: True


## 13. Cleanup

`unset_connection()` clears the settings and closes every connection, the embedded engine included:
its in-memory database is discarded.

In [18]:
await SurrealDBConnectionManager.unset_connection()
print("Connection configured:", SurrealDBConnectionManager.is_connection_set())

Connection configured: False
